In [2]:
import json
from collections import Counter
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "data").exists():
    ROOT = ROOT.parent

REPORT_JSON = ROOT / "data" / "report.json"
REPORT_MD = ROOT / "data" / "report.md"

print("Repo root:", ROOT)
print("Reading:", REPORT_JSON)
print("Writing:", REPORT_MD)

Repo root: c:\Users\sophi\OneDrive\Documents\Aerospace-1B-compliance-copilot
Reading: c:\Users\sophi\OneDrive\Documents\Aerospace-1B-compliance-copilot\data\report.json
Writing: c:\Users\sophi\OneDrive\Documents\Aerospace-1B-compliance-copilot\data\report.md


In [3]:
with open(REPORT_JSON, "r", encoding="utf-8") as f:
    report = json.load(f)

print("Loaded report with:")
print("-", len(report["requirement_reviews"]), "requirement reviews")
print("-", len(report["relationships"]), "relationships")
print("-", len(report["document_findings"]), "document findings")
print("-", len(report["swe_coverage"]), "SWE coverage entries")

Loaded report with:
- 54 requirement reviews
- 3 relationships
- 5 document findings
- 6 SWE coverage entries


In [4]:
quality_counts = Counter(
    review["quality_status"]
    for review in report["requirement_reviews"]
)

issue_counts = Counter(
    issue["issue_code"]
    for review in report["requirement_reviews"]
    for issue in review["quality_issues"]
)

print("Quality status counts:")
print(quality_counts)

print("\nIssue code counts:")
print(issue_counts)

Quality status counts:
Counter({'NEEDS_REVIEW': 30, 'ACCEPTABLE': 24})

Issue code counts:
Counter({'UNDEFINED_TERM': 16, 'VAGUE_CRITERION': 14, 'SCOPE_AMBIGUITY': 4, 'ALLOCATION_UNCLEAR': 3, 'UNSPECIFIED_RESPONSE': 2, 'SAFETY_INTERACTION': 1, 'UNVERIFIABLE_ABSOLUTE': 1, 'ESCAPE_CLAUSE': 1, 'OTHER': 1})


In [5]:
def md_link(label, url):
    if not url:
        return label
    return f"[{label}]({url})"


def format_nasa_reference(ref):
    parts = []

    if ref.get("document"):
        parts.append(ref["document"])

    if ref.get("swe_id"):
        parts.append(ref["swe_id"])

    if ref.get("section"):
        parts.append(ref["section"])

    label = " | ".join(parts)

    return md_link(label, ref.get("url"))


def format_evidence(evidence):
    locator = evidence.get("locator", "")
    excerpt = evidence.get("excerpt", "")

    if locator and excerpt:
        return f"- **{locator}:** \"{excerpt}\""
    elif excerpt:
        return f"- \"{excerpt}\""
    elif locator:
        return f"- **{locator}**"
    return "-"

In [6]:
lines = []

run = report["run"]

# ------------------------------------------------------------
# Header
# ------------------------------------------------------------

lines.append("# Software Requirements Compliance Review")
lines.append("")

lines.append(f"- **Document ID:** {run['input_document_id']}")
lines.append(f"- **Document version:** {run['input_document_version']}")
lines.append(f"- **Evaluation scope:** {run['evaluation_scope']}")
lines.append(f"- **Generated at (UTC):** {run['generated_at_utc']}")
lines.append(f"- **Method:** {run['method']}")
lines.append("")

lines.append("## Limitations")
lines.append("")

for limitation in run["limitations"]:
    lines.append(f"- {limitation}")

lines.append("")

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

lines.append("## Summary")
lines.append("")

lines.append("### Requirements by quality status")
lines.append("")

lines.append("| Quality status | Count |")
lines.append("|---|---:|")

for status, count in sorted(quality_counts.items()):
    lines.append(f"| {status} | {count} |")

lines.append("")

lines.append("### Issue code frequency")
lines.append("")

lines.append("| Issue code | Count |")
lines.append("|---|---:|")

for issue_code, count in sorted(issue_counts.items()):
    lines.append(f"| {issue_code} | {count} |")

lines.append("")

# ------------------------------------------------------------
# All requirements
# ------------------------------------------------------------

lines.append("## Requirement Review Summary")
lines.append("")

lines.append("| Requirement ID | Quality status | Issue codes | Primary SWE |")
lines.append("|---|---|---|---|")

for review in report["requirement_reviews"]:
    issue_codes = ", ".join(
        issue["issue_code"]
        for issue in review["quality_issues"]
    ) or "None"

    primary_swe = review["primary_swe_id"] or "None"

    lines.append(
        f"| {review['requirement_id']} "
        f"| {review['quality_status']} "
        f"| {issue_codes} "
        f"| {primary_swe} |"
    )

lines.append("")

# ------------------------------------------------------------
# Detailed requirement findings
# ------------------------------------------------------------

lines.append("## Requirement Details")
lines.append("")

for review in report["requirement_reviews"]:
    if review["quality_status"] not in {"NEEDS_REVIEW", "UNDETERMINED"}:
        continue

    lines.append(f"### {review['requirement_id']}")
    lines.append("")
    lines.append(f"**Requirement text:** {review['raw_text']}")
    lines.append("")
    lines.append(f"**Quality status:** {review['quality_status']}")
    lines.append("")

    for issue in review["quality_issues"]:
        lines.append(f"#### {issue['issue_code']}")
        lines.append("")
        lines.append(f"- **Severity:** {issue['severity']}")
        lines.append(f"- **Explanation:** {issue['explanation']}")
        lines.append(
            f"- **Requires stakeholder input:** "
            f"{issue['requires_stakeholder_input']}"
        )
        lines.append(f"- **Recommendation:** {issue['recommendation']}")
        lines.append("")

        lines.append("**SRS evidence:**")
        lines.append("")

        for evidence in issue["srs_evidence"]:
            lines.append(format_evidence(evidence))

        lines.append("")
        lines.append("**NASA references:**")
        lines.append("")

        for ref in issue["nasa_references"]:
            lines.append(f"- {format_nasa_reference(ref)}")

        lines.append("")

# ------------------------------------------------------------
# Relationships
# ------------------------------------------------------------

lines.append("## Relationships")
lines.append("")

for rel in report["relationships"]:
    req_ids = ", ".join(rel["requirement_ids"])

    lines.append(f"### {rel['relationship_id']}")
    lines.append("")
    lines.append(f"- **Requirements:** {req_ids}")
    lines.append(f"- **Status:** {rel['status']}")
    lines.append(f"- **Issue code:** {rel['issue_code']}")
    lines.append(f"- **Explanation:** {rel['explanation']}")
    lines.append(f"- **Recommendation:** {rel['recommendation']}")
    lines.append("")

    lines.append("**SRS evidence:**")
    lines.append("")

    for evidence in rel["srs_evidence"]:
        lines.append(format_evidence(evidence))

    lines.append("")
    lines.append("**NASA references:**")
    lines.append("")

    for ref in rel["nasa_references"]:
        lines.append(f"- {format_nasa_reference(ref)}")

    lines.append("")

# ------------------------------------------------------------
# Document findings
# ------------------------------------------------------------

lines.append("## Document Findings")
lines.append("")

for finding in report["document_findings"]:
    lines.append(f"### {finding['finding_id']}: {finding['criterion']}")
    lines.append("")
    lines.append(f"- **Status:** {finding['status']}")
    lines.append(f"- **Issue code:** {finding['issue_code']}")
    lines.append(f"- **Searched scope:** {finding['searched_scope']}")
    lines.append(f"- **Explanation:** {finding['explanation']}")
    lines.append(f"- **Recommendation:** {finding['recommendation']}")
    lines.append("")

    lines.append("**SRS evidence:**")
    lines.append("")

    for evidence in finding["srs_evidence"]:
        lines.append(format_evidence(evidence))

    lines.append("")
    lines.append("**NASA references:**")
    lines.append("")

    for ref in finding["nasa_references"]:
        lines.append(f"- {format_nasa_reference(ref)}")

    lines.append("")

# ------------------------------------------------------------
# SWE coverage
# ------------------------------------------------------------

lines.append("## SWE Coverage")
lines.append("")


for swe in report["swe_coverage"]:
    lines.append(f"### {swe['swe_id']}")
    lines.append("")
    lines.append(f"- **Document evidence:** {swe['document_evidence']}")
    lines.append(f"- **Project compliance:** {swe['project_compliance']}")
    lines.append(f"- **Rationale:** {swe['rationale']}")
    lines.append("")

    supported = swe["supported_by_requirement_ids"]
    if supported:
        lines.append(
            "- **Supported by requirements:** "
            + ", ".join(supported)
        )
    else:
        lines.append("- **Supported by requirements:** None")

    lines.append("")

    lines.append("**SRS evidence:**")
    lines.append("")

    for evidence in swe["srs_evidence"]:
        lines.append(format_evidence(evidence))

    lines.append("")

    lines.append("**External evidence needed:**")
    lines.append("")

    for item in swe["external_evidence_needed"]:
        lines.append(f"- {item}")

    lines.append("")

    lines.append("**NASA reference:**")
    lines.append("")

    lines.append(
        f"- {format_nasa_reference(swe['nasa_reference'])}"
    )

    lines.append("")

In [7]:
markdown_text = "\n".join(lines)

with open(REPORT_MD, "w", encoding="utf-8") as f:
    f.write(markdown_text)

print(f"Wrote Markdown report to: {REPORT_MD}")

Wrote Markdown report to: c:\Users\sophi\OneDrive\Documents\Aerospace-1B-compliance-copilot\data\report.md


In [8]:
assert REPORT_MD.exists()

for review in report["requirement_reviews"]:
    assert review["requirement_id"] in markdown_text

assert len(report["requirement_reviews"]) == 54
assert len(report["relationships"]) == 3
assert len(report["document_findings"]) == 5
assert len(report["swe_coverage"]) == 6

print("Sanity checks passed.")

Sanity checks passed.
